In [114]:
import requests
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import json
import os
import pandas as pd
import numpy as np
from pandas import DataFrame 

# Mongo 
uri = os.getenv('SBS_V1_MONGO_URI')

# rapidApi
headers = {
    "X-RapidAPI-Key": os.getenv('RAPID_API_KEY'),
    "X-RapidAPI-Host": os.getenv('RAPID_API_HOST')
}

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))
db = client["SBSV1"]
collection = db["nba_games_historical"]

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [115]:
#########################################################
# games by game ids #####################################
def get_team_nicknames_by_id():
    # team code by id
    url = "https://api-nba-v1.p.rapidapi.com/teams"
    response = requests.get(url, headers=headers).json()["response"]
    df = pd.DataFrame(response)
    df = df.loc[(df["nbaFranchise"] == True) & (df["allStar"] == False)]
    
    team_nickname_to_id_map = {}
    
    for index, row in df.iterrows():
        team_nickname_to_id_map.update({row["nickname"]: row["id"]})
    return team_nickname_to_id_map
#########################################################

In [116]:
cols_to_drop_for_game_stats = [
    "stage",
    "officials",
    "timesTied",
    "leadChanges",
    "nugget",
    "date.end",
    "date.duration",
    "status.clock",
    "status.halftime",
    "status.short",
    "status.long",
    "periods.current",
    "periods.total",
    "periods.endOfPeriod",
    "arena.name",
    "arena.city",
    "arena.state",
    "arena.country",
    "teams.home.logo",
    "scores.home.series.win",
    "scores.home.series.loss",
    "scores.home.win",
    "scores.home.loss",
    "teams.visitors.logo",
    "scores.visitors.series.win",
    "scores.visitors.series.loss",
    "scores.visitors.win",
    "scores.visitors.loss",
]

#########################################################
# drop all cols from a df ###############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [117]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team_id):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team_id}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

In [118]:
teams = get_team_nicknames_by_id()
#########################################################
# load games data into nba_games_mongo_historical #######
def load_nba_games(season):    
    games_data_dict = []
    for team in teams:
        games_df = get_games_by_game_ids(season, teams.get(team))
        games_df['_id'] = games_df['id']
        games_data_dict.append(games_df.to_dict("records"))

    flat_games_data_dict = []
    for row in games_data_dict:
        flat_games_data_dict.extend(row)

    deduped_dict = {}
    for item in flat_games_data_dict:
        deduped_dict[item["_id"]] = item
    flat_games_data_dict = list(deduped_dict.values())
    
    game_data_to_insert = []
    for d in flat_games_data_dict:
        game_data_to_insert.append(rename_and_remove_fields(d))

    # Insert the data into the MongoDB collection
    result = collection.insert_many(game_data_to_insert)

    # Print the inserted IDs
    print("Inserted IDs:", result.inserted_ids)
#########################################################

In [119]:
cols_to_drop_for_player_stats = [
    "comment",
    "team.nickname",
    "team.code",
    "team.name",
    "team.logo",
]

#########################################################
# get game ids for season ###############################
def get_game_for_season(season):
    return collection.find({ "season": season })
#########################################################

#########################################################
# insert player stats for each game #####################
def get_players_per_game_df(game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    return drop_cols(df, cols_to_drop_for_player_stats)
#########################################################

In [120]:
from collections import defaultdict

#########################################################
# load player game data into player_game_stats_historical
def load_player_game_stats(season):
    games = get_game_for_season(season)
    all_player_stats_per_game = []
    for game in games:
        game_id = game['_id']
        home_team_id = game['teamsHomeId']
        visitors_team_id = game['teamsVisitorsId']

        players_dict_list = get_players_per_game_df(game_id).to_dict("records")

        # Initialize a defaultdict
        players_grouped_by_team = defaultdict(list)
        # Group the data
        for item in players_dict_list:
            players_grouped_by_team[item['team.id']].append(item)
        players_grouped_by_team['teamsHomePlayers'] = players_grouped_by_team.pop(home_team_id)
        players_grouped_by_team['teamsVisitorsPlayers'] = players_grouped_by_team.pop(visitors_team_id)
        players_grouped_by_team['teamsHomeId'] = home_team_id
        players_grouped_by_team['teamsVisitorsId'] = visitors_team_id
        players_grouped_by_team['season'] = season
        players_grouped_by_team['dateStart'] = game['dateStart']
        players_grouped_by_team['_id'] = game_id
        all_player_stats_per_game.append(rename_and_remove_fields(players_grouped_by_team))
    
    players_game_stats_historical_collection = db["player_game_stats_historical"]
    result = players_game_stats_historical_collection.insert_many(all_player_stats_per_game)
    
    # Print the inserted IDs
    print("Inserted IDs:", result.inserted_ids)
#########################################################

In [121]:
import re

#########################################################
# Function to convert dot-separated to camelCase
def to_camel_case(s):
    parts = s.split('.')
    return parts[0] + ''.join(word.capitalize() for word in parts[1:])
#########################################################

#########################################################
# Function to recursively rename fields in a document and remove fields with periods
def rename_and_remove_fields(doc):
    if isinstance(doc, dict):
        new_doc = {}
        for key, value in doc.items():
            if '.' in key:
                new_key = to_camel_case(key)
                if isinstance(value, (dict, list)):
                    value = rename_and_remove_fields(value)
                new_doc[new_key] = value
            else:
                if isinstance(value, (dict, list)):
                    value = rename_and_remove_fields(value)
                new_doc[key] = value
        return new_doc
    elif isinstance(doc, list):
        return [rename_and_remove_fields(item) for item in doc]
    return doc
#########################################################

#########################################################
def rename_all_fields_in_collection(collection):
    # Retrieve all documents from the collection
    documents = collection.find()

    # Update each document
    for doc in documents:
        # Get the document ID
        doc_id = doc["_id"]

        # Rename and remove fields
        updated_doc = rename_and_remove_fields(doc)

        # Remove old fields that contain periods
        update_operations = {
            "$set": updated_doc,
            "$unset": {key: "" for key in doc.keys() if '.' in key}
        }

        # Save the updated document back to the collection
        collection.update_one({"_id": doc_id}, update_operations)

    print("Fields containing periods renamed to camelCase or removed successfully.")
#########################################################

#########################################################
def get_dot_separated_keys(document):
    dot_keys = [key for key in document if '.' in key]
    return dot_keys
#########################################################

#########################################################
def remove_all_dot_separated_keys(collection):
    # Find all documents in the collection
    documents = collection.find()

    for document in documents:
        doc_id = document['_id']
        dot_keys = get_dot_separated_keys(document)

        if dot_keys:
            unset_query = {key: "" for key in dot_keys}
            # Remove the dot-separated keys from the document
            collection.update_one({'_id': doc_id}, {'$unset': unset_query})

    print("Dot-separated keys have been removed.")
#########################################################   


In [122]:
# players_game_stats_historical_collection = db["player_game_stats_historical"]
# nba_games_historical_collection = db["nba_games_historical"]

# # rename_all_fields_in_collection(players_game_stats_historical_collection)
# # rename_all_fields_in_collection(nba_games_historical_collection)

# remove_all_dot_separated_keys(players_game_stats_historical_collection)
# remove_all_dot_separated_keys(nba_games_historical_collection)

In [123]:
# load_player_game_stats(2023)

In [124]:
#########################################################
# update entire collection ##############################
def update_entire_collection(collection, update):
    result = collection.update_many({}, update)
    print(result)
#########################################################

In [125]:
# players_game_stats_historical_collection = db["player_game_stats_historical"]
# update_entire_collection(players_game_stats_historical_collection, { '$set': { 'season': 2023 }})

In [149]:
#########################################################
# get player per team and season ########################  
def get_player_per_team_and_season(team, season):
    url = "https://api-nba-v1.p.rapidapi.com/players"
    querystring = {"team": team,"season": season }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    return df
#########################################################

#########################################################
# get mongo pipeline to load player stats per season ####
def get_mongo_pipeline_for_player_stats_per_season(team_id, player_id, season, date_start, date_end):
    return [
                {
                    '$match': { 
                        "$or": [
                            {"teamsHomeId": team_id},
                            {"teamsVisitorsId": team_id}
                        ], 
                        "$or": [
                            {"teamsHomePlayers": {"$elemMatch": {"playerId": player_id}}},
                            {"teamsVisitorsPlayers": {"$elemMatch": {"playerId": player_id}}}
                        ],
                        'season': season
                    }  
                },
                {
                    '$unwind': '$teamsHomePlayers'
                },
                {
                    '$match': {
                        'teamsHomePlayers.playerId': player_id
                    }
                },
                {
                    '$project': {
                        'playerStats': '$teamsHomePlayers',
                        'dateStart': 1
                    }
                },
                {
                    '$unionWith': {
                        'coll': 'player_game_stats_historical',
                        'pipeline': [
                            {
                                '$match': { 
                                    "$or": [
                                        {"teamsHomeId": team_id},
                                        {"teamsVisitorsId": team_id}
                                    ], 
                                    "$or": [
                                        {"teamsHomePlayers": {"$elemMatch": {"playerId": player_id}}},
                                        {"teamsVisitorsPlayers": {"$elemMatch": {"playerId": player_id}}}
                                    ],
                                    'season': season  
                                }
                            },
                            {
                                '$unwind': '$teamsVisitorsPlayers'
                            },
                            {
                                '$match': {
                                    'teamsPlayers.playerId': player_id
                                }
                            },
                            {
                                '$project': {
                                    'playerStats': '$teamsVisitorsPlayers',
                                    'dateStart': 1
                                }
                            }
                        ]
                    }
                },
                {
                    '$match': { 
                        '$and': [
                            { 
                                'dateStart': { 
                                    '$gte': date_start
                                }
                            }, 
                            { 
                                'dateStart': {
                                    '$lte': date_end
                                }
                            }
                        ] 
                    }  
                },
                {
                    '$sort': {
                        'dateStart': 1
                    }
                }
            ]
#########################################################


player_statistical_columns = [
    'playerStats.points',
    'playerStats.min', 
    'playerStats.fgm', 
    'playerStats.fga',
    'playerStats.fgp', 
    'playerStats.ftm', 
    'playerStats.fta',
    'playerStats.ftp', 
    'playerStats.tpm', 
    'playerStats.tpa',
    'playerStats.tpp', 
    'playerStats.offReb', 
    'playerStats.defReb',
    'playerStats.totReb', 
    'playerStats.assists', 
    'playerStats.pFouls',
    'playerStats.steals', 
    'playerStats.turnovers', 
    'playerStats.blocks',
    'playerStats.plusMinus'
]

#########################################################
# rename player stats cols ##############################
def rename_player_stats_cols(col, new_prefix):
    l = col.split('.')
    return f"{new_prefix}.{l[1]}"
#########################################################

#########################################################
# transform players stats df ############################
def transform_player_stats_df(df):
    l = col.split('.')
    return f"{new_prefix}.{l[1]}"
#########################################################

#########################################################
# calculate player rolling stats averages ###############
def calculate_player_rolling_averages(df, window):
    # Calculate the rolling average for the selected columns
    rolling_avgs = df[player_statistical_columns].rolling(window=window, min_periods=1).mean()

    # Rename the columns to indicate they are rolling averages
    rolling_avgs = rolling_avgs.rename(columns=lambda x: rename_player_stats_cols(x, f"rollingAvg{window}"))
    
    return rolling_avgs
#########################################################

#########################################################
# calculate player expanding stats averages #############
def calculate_player_expanding_averages(df):
    # Calculate the rolling average for the selected columns
    expanding_avgs = df[player_statistical_columns].expanding().mean()

    # Rename the columns to indicate they are rolling averages
    expanding_avgs = expanding_avgs.rename(columns=lambda x: rename_player_stats_cols(x, 'expandingAvg'))
    
    return expanding_avgs
#########################################################

nba_season_2023_start_date = '2023-10-24'
nba_season_2023_end_date = '2024-04-14'
    
#########################################################
# load player avgs per game #############################
def load_player_avgs_through_season(season, start_date, end_date):
    player_avgs_game_stats = db["player_game_stats_avgs_historical"]
    teams = get_team_nicknames_by_id()
    for team_nickname, team_id in teams.items():
        players_game_stats_historical_collection = db["player_game_stats_historical"]
        player_ids = get_player_per_team_and_season(team_id, season)['id']
        for player_id in player_ids:
            pipeline = get_mongo_pipeline_for_player_stats_per_season(team_id, player_id, season, start_date, end_date)
            player_games = players_game_stats_historical_collection.aggregate(pipeline)
            normalized_df = pd.json_normalize(list(player_games))
            expanding_avg_df = calculate_player_expanding_averages(normalized_df)
            rolling_avg_5_df = calculate_player_rolling_averages(normalized_df, 5)
            rolling_avg_10_df = calculate_player_rolling_averages(normalized_df, 10)
            full_player_avgs_df = pd.concat([normalized_df, expanding_avg_df, rolling_avg_5_df, rolling_avg_10_df], axis=1)
            
            nested_json = [nest_dict(record) for record in full_player_avgs_df.to_dict(orient='records')]
            return player_avgs_game_stats.insert_many(nested_json)
#########################################################

In [150]:
import json

# Function to convert flat dictionary to nested dictionary
def nest_dict(flat_dict):
    nested_dict = {}
    for key, value in flat_dict.items():
        parts = key.split('.')
        d = nested_dict
        for part in parts[:-1]:
            if part not in d:
                d[part] = {}
            d = d[part]
        d[parts[-1]] = value
    return nested_dict

# Example
# nested_json = [nest_dict(record) for record in df.to_dict(orient='records')]


In [151]:
res = load_player_avgs_through_season(2023, nba_season_2023_start_date, nba_season_2023_end_date)
# nested_json = [nest_dict(record) for record in df.to_dict(orient='records')]
# print(nested_json)

'[\n  {\n    "_id": 12567,\n    "dateStart": "2023-10-27T23:30:00.000Z",\n    "playerStats": {\n      "points": 18,\n      "pos": "SG",\n      "min": "35",\n      "fgm": 8,\n      "fga": 17,\n      "fgp": "47.1",\n      "ftm": 2,\n      "fta": 3,\n      "ftp": "66.7",\n      "tpm": 0,\n      "tpa": 3,\n      "tpp": "0",\n      "offReb": 2,\n      "defReb": 5,\n      "totReb": 7,\n      "assists": 6,\n      "pFouls": 1,\n      "steals": 1,\n      "turnovers": 1,\n      "blocks": 1,\n      "plusMinus": "-6",\n      "playerId": 382,\n      "playerFirstname": "Dejounte",\n      "playerLastname": "Murray",\n      "teamId": 1,\n      "gameId": 12567\n    },\n    "expandingAvg": {\n      "points": 18.0,\n      "min": 35.0,\n      "fgm": 8.0,\n      "fga": 17.0,\n      "fgp": 47.1,\n      "ftm": 2.0,\n      "fta": 3.0,\n      "ftp": 66.7,\n      "tpm": 0.0,\n      "tpa": 3.0,\n      "tpp": 0.0,\n      "offReb": 2.0,\n      "defReb": 5.0,\n      "totReb": 7.0,\n      "assists": 6.0,\n      "pFo